# Data enrichement Pipeline for geometric data

## 1. Extracting labels from IFC files and save as parquet file

In [1]:
# imports
import ifcopenshell
from pathlib import Path
import pandas as pd
import sys

# helpers
sys.path.insert(0, "../../")
from geometric_extraction_helper import (
    ALL_FEATURE_KEYS,
    _get_settings,
    iter_elements_with_features
)

# global settings for geometry extraction
SETTINGS = _get_settings()

/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# set directories and paths
IFC_MODELS_DIR = "../../0_data/1_ifc_models"
DATA_DIR  = "../3_2_analyze_labels/data_eBKPh_labels_renamed.parquet"
OUTPUT_PATH = "./data_geometric_featueres_enriched.parquet"

# ifc_file_path values were stored relative to collecting_data.ipynb,
IFC_BASE_DIR = Path("../../1_data_mapping/1_1_data_collection")

In [3]:
# load existing dataset
df = pd.read_parquet(DATA_DIR)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns from {DATA_DIR}")

Loaded 71,922 rows, 124 columns from ../3_2_analyze_labels/data_eBKPh_labels_renamed.parquet


In [4]:
# pre-allocate geometric feature columns with NaN
for col in ALL_FEATURE_KEYS:
    df[col] = float("nan")

# print how many unique IFC file paths we have to enrich
unique_paths = df["ifc_file_path"].dropna().unique()
print(f"Enriching {len(df):,} rows from {len(unique_paths)} IFC files.\n")

Enriching 71,922 rows from 35 IFC files.



In [5]:
# iterate over unique IFC file paths for extracting geometric features and merge it
for raw_path in unique_paths:
    ifc_path = Path(raw_path)

    # if the path is not absolute, resolve it relative to the base dir
    if not ifc_path.is_absolute():
        ifc_path = (IFC_BASE_DIR / raw_path).resolve()

    # check if the IFC file exists, if not print a warning and skip
    if not ifc_path.exists():
        print(f"File not found, skipping: {ifc_path}")
        continue

    print(f"Processing: {ifc_path.name}")
    ifc_file = ifcopenshell.open(str(ifc_path))

    # collect features for all elements in this model as {guid: feature_dict}
    guid_features = {}
    for element, features in iter_elements_with_features(ifc_file, settings=SETTINGS):
        guid_features[element.GlobalId] = features

    if not guid_features:
        continue

    # build patch DataFrame and merge into the main DataFrame using guid
    patch = pd.DataFrame.from_dict(guid_features, orient="index")
    patch.index.name = "guid"
    patch = patch.reset_index()

    # create a boolean mask for rows with the current IFC file path and merge the patch on guid
    mask = df["ifc_file_path"] == raw_path
    merged = df.loc[mask, ["guid"]].merge(patch, on="guid", how="left")
    df.loc[mask, ALL_FEATURE_KEYS] = merged[ALL_FEATURE_KEYS].values

Processing: ZUST_P_ARC_VOL_.ifc
Processing: ZUST_P_ARC_EBK_.ifc
Processing: ZUST_C_ARC_VOL_.ifc
Processing: ZUST_C_ARC_NGF_.ifc
Processing: ZUST_C_ARC_EBK_.ifc
Processing: ZUST_B_ARC_VOL_.ifc
Processing: ZUST_B_ARC_NGF_.ifc
Processing: ZUST_B_ARC_EBK_.ifc
Processing: ZUST_A_ARC_VOL_.ifc
Processing: ZUST_A_ARC_NGF_.ifc
Processing: ZUST_A_ARC_EBK_.ifc
Processing: ZUST_ARC_NGF_.ifc
Processing: ZEGA_52_ARC_eBKPh_260127.ifc
Processing: RALU_32_ARC_eBKPh.ifc
Processing: LUMU_31_ARC_eBKPh_V3.ifc
Processing: KEPR_31_ARC_eBKPh_260209.ifc
Processing: KEHO_31_ARC_eBKPh_260127.ifc
Processing: IMBU_52_ARC_eBKPh_260127.ifc
Processing: GSRH_31_ARC_eBKPh_V2.ifc
Processing: GSRH_31_ARC_eBKPh_V1.ifc
Processing: GERB_31_ARC_eBKPh_250221.ifc
Processing: GERB_31_ARC_eBKPh_241209.ifc
Processing: ESAK_21_ARC_eBKPh.ifc
Processing: CHLI_51_AVM_Haus B.ifc
Processing: CHLI_51_AVM_Haus A.ifc
Processing: CHLI_51_ANG_Haus B.ifc
Processing: CHLI_51_ANG_Haus A.ifc
Processing: CHLI_51_ALE_Haus B_eBKP-H.ifc
Processing:

In [6]:
# save enriched dataset
df.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved {len(df):,} rows to the path: {OUTPUT_PATH}")


Saved 71,922 rows to the path: ./data_geometric_featueres_enriched.parquet


In [7]:
 # print how many non-NaN values we have for each geometric feature
df[ALL_FEATURE_KEYS].notna().sum().to_frame("non_nan_count")

,non_nan_count
aabb_min_x,71897
aabb_min_y,71897
aabb_min_z,71897
aabb_max_x,71897
aabb_max_y,71897
...,...
mat_putz,71922
mat_stahl,71922
mat_zement,71922
horizontal_elements_above,13346


In [8]:
 # print how many non nan values we have for each geometric feature
df[ALL_FEATURE_KEYS].value_counts()

aabb_min_x  aabb_min_y  aabb_min_z  aabb_max_x  aabb_max_y  aabb_max_z  aabb_len_x  aabb_len_y  aabb_len_z  aabb_ratio_z_x  aabb_ratio_z_y  aabb_ratio_x_y  aabb_diagonal  aabb_volume  geom_volume  geom_surface_area  geom_projected_area  geom_centroid_x  geom_centroid_y  geom_centroid_z  geom_z_min  geom_z_max  geom_z_range  geom_ratio_area_vol  geom_compactness  geom_layer_count  tfbb_extent_0  tfbb_extent_1  tfbb_extent_2  tfbb_volume  tfbb_ratio_01  tfbb_ratio_02  tfbb_ratio_12  tfbb_linearity  tfbb_planarity  tfbb_sphericity  tfbb_primary_ax_x  tfbb_primary_ax_y  tfbb_primary_ax_z  topo_vertex_count  topo_face_count  topo_unique_edge_count  topo_euler_characteristic  topo_genus  topo_max_face_area  topo_avg_face_area  topo_vertex_edge_ratio  topo_connected_components  mat_allgemein  mat_aluminium  mat_backstein  mat_bekleidung  mat_belag  mat_beton  mat_dämm  mat_foamglas  mat_gips  mat_glas  mat_holz  mat_werkstoff  mat_kalksandstein  mat_keramik  mat_kies  mat_kunststein  mat_kuns

In [9]:
df["mat_zement"].value_counts()

mat_zement
0.0    71762
1.0      160
Name: count, dtype: int64

In [10]:
df["mat_backstein"].value_counts()

mat_backstein
0.0    67800
1.0     4122
Name: count, dtype: int64

In [11]:
df["horizontal_elements_above"].value_counts()

horizontal_elements_above
0.0      1676
22.0      352
6.0       328
14.0      324
18.0      307
         ... 
229.0       1
245.0       1
210.0       1
203.0       1
254.0       1
Name: count, Length: 330, dtype: int64

In [12]:
df["horizontal_elements_below"].value_counts()

horizontal_elements_below
0.0      1332
14.0      353
6.0       352
12.0      339
10.0      298
         ... 
355.0       1
333.0       1
290.0       1
226.0       1
235.0       1
Name: count, Length: 342, dtype: int64